# FTTH QoE Risk - Process Phase (GitHub - Absolute Paths)

Notebook reproducible usando `pathlib` absoluto. Siempre resuelve la raíz del proyecto, no la raíz del disco (`/data/`).

- Local: `D:\Capstone FFTH\data/`
- GitHub Linux: `/home/runner/work/repo/repo/data/`


In [ ]:
from pathlib import Path
import pandas as pd

# --- Resolución absoluta de la raíz del proyecto ---
current = Path.cwd().resolve()

# Busca hacia arriba la carpeta que contenga 'data' y 'python' (raíz del repo)
ROOT = None
for parent in [current] + list(current.parents):
    if (parent / "data").exists():
        ROOT = parent
        break

# Fallback: si no encuentra, si estoy en python/ sube uno
if ROOT is None:
    ROOT = current.parent if current.name == "python" else current

BASE = ROOT / "data"
PYTHON_DIR = ROOT / "python"

print(f"current: {current}")
print(f"ROOT proyecto (absoluta): {ROOT}")
print(f"BASE (absoluta): {BASE}")
print(f"PYTHON_DIR (absoluta): {PYTHON_DIR}")
print(f"BASE existe: {BASE.exists()}")


In [ ]:
def p(fname: str) -> Path:
    return BASE / fname

df_ping = pd.read_csv(p("sample1000_curr_ping.csv"))
df_udp = pd.read_csv(p("sample1000_udplatency.csv"))
df_fiber = pd.read_csv(p("fiber_units.csv"))

print(f"ping: {df_ping.shape} | udp: {df_udp.shape} | fiber: {df_fiber.shape}")
df_ping.head(3)


### 1. packet_loss_pct = failures / (successes+failures)*100

In [ ]:
df_ping["packet_loss_pct"] = df_ping["failures"] / (df_ping["successes"]+df_ping["failures"]) * 100
df_ping_fiber = df_ping.merge(df_fiber[["unit_id"]], on="unit_id", how="inner")
df_ping_fiber[["unit_id","failures","successes","packet_loss_pct"]].head()


### 2. is_peak_hour = 1 si 19-23h (hora Netflix)

In [ ]:
df_ping_fiber["hour"] = pd.to_datetime(df_ping_fiber["dtime"], errors='coerce').dt.hour
df_ping_fiber["is_peak_hour"] = ((df_ping_fiber["hour"]>=19) & (df_ping_fiber["hour"]<=23)).astype(int)
df_ping_fiber[["dtime","hour","is_peak_hour"]].head()


### 3. rtt_p95 = PERCENTILE_CONT(0.95) por unit_id, ddate

In [ ]:
df_udp_fiber = df_udp.merge(df_fiber[["unit_id"]], on="unit_id", how="inner")
rtt_p95 = df_udp_fiber.groupby(["unit_id","ddate"])["rtt_avg"].quantile(0.95).reset_index().rename(columns={"rtt_avg":"rtt_p95"})
rtt_p95.head()


### 4. risk_flag = 1 si (packet_loss>1.5 o rtt_p95>80) y peak hour

In [ ]:
df_final = df_ping_fiber.merge(rtt_p95, on=["unit_id","ddate"], how="left")
df_final["risk_flag"] = (((df_final["packet_loss_pct"]>1.5) | (df_final["rtt_p95"]>80)) & (df_final["is_peak_hour"]==1)).astype(int)
print(df_final["risk_flag"].value_counts(normalize=True))
df_final.head()


In [ ]:
out_path = BASE / "risk_flag_sample.csv"
df_final.to_csv(out_path, index=False)
print(f"Exportado en ruta absoluta: {out_path}")
print(f"Existe: {out_path.exists()}")
